## Notebook for the research paper: Fooling Abusive Text Detection using XAI Methods.
Authors: group7 Trustworthy and Explainable AI.

#### Imports and pytorch GPU check

In [223]:
# Imports
import io
import json
import nlpaug.augmenter.word as naw
import numpy as np
import os
import pandas as pd
import pickle
import random
import re
import shap
import torch

from contextlib import redirect_stdout
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TextClassificationPipeline
from lime.lime_text import LimeTextExplainer

# To download glove data, uncomment the 2 lines below
# from nlpaug.util.file.download import DownloadUtil
# DownloadUtil.download_glove(model_name='glove.6B', dest_dir=os.path.join(os.getcwd(), 'glove_model')) # Download GloVe model



print('GPU enabled: ', torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

GPU enabled:  True


#### Parameters

In [224]:
# Param space

model_path           = os.path.join(os.getcwd(), "HateBERT_fine_tuned_models", "HateBERT_abuseval") # Copy your files here

offenseval_path      = os.path.join(os.getcwd(), 'DATASET_data', 'offenseval_data') # offenseval dataset
abuseval_path        = os.path.join(os.getcwd(), 'DATASET_data', 'abuseval_labels') # abuseval labels

samples_path   = os.path.join(offenseval_path, 'olid-training-v1.0.tsv') # train dataset
labels_path    = os.path.join(abuseval_path, 'abuseval_offenseval_train.tsv') # train dataset labels


#### Basic Function Definitions

In [225]:
# Function space

# read saved json data as follows (returns numpy matrix)
def read_json(file_path):
    with open(file_path, 'r') as f:
        return np.array(json.load(f))

    # read saved json data as follows (returns numpy matrix)
def read_json(file_path):
    with open(file_path, 'r') as f:
        return json.load(f)

def read_lime(filename, n_files=1):
    lime_data = []              # cant be set to numpy as it has complex shape
    if n_files <= 1:
        lime_data.extend(read_json(os.path.join(os.getcwd(), f'{filename}.json')))
        return lime_data
    
    for part in range(n_files): # loop over amount of split parts
        lime_data.extend(read_json(os.path.join(os.getcwd(), f'{filename}{part+1}.json')))
    return lime_data

def read_model(filename, n_files=1):
    model_data = []     # will be converted to numpy
    if n_files <= 1:
        model_data.extend(read_json(os.path.join(os.getcwd(), f'{filename}.json')))
        model_data = np.array(model_data, dtype=object)
        return model_data

    for part in range(n_files): # loop over amount of split parts
        model_data.extend(read_json(os.path.join(os.getcwd(), f'{filename}{part+1}.json')))
    model_data = np.array(model_data, dtype=object)
    return model_data

def read_shap(filename, n_files=1):
    shap_data = []      # cant be set to numpy as it has complex shape
    if n_files <= 1:
        shap_data.extend(read_json(os.path.join(os.getcwd(), f'{filename}.json')))
        return shap_data
    
    for part in range(n_files): # loop over amount of split parts
        shap_data.extend(read_json(os.path.join(os.getcwd(), f'{filename}{part+1}.json')))
    return shap_data

def read_predictions_perturbed(addition):
    perturb_data = read_json(os.path.join(os.getcwd(), f'data_model_perturbed_{addition}.json'))
    perturb_data = np.array(perturb_data, dtype=object)
    return perturb_data

#### Data and Model Loading

In [226]:
# Data & model loader
model = AutoModelForSequenceClassification.from_pretrained(model_path)  # auto selects the correct base model for the pretrained weights of HateBERT
tokenizer = AutoTokenizer.from_pretrained(model_path)                   # auto selects the correct tokenizer
model.eval()
model.to(device)
pipeline = TextClassificationPipeline(model=model, tokenizer=tokenizer, return_all_scores=True, device=device)

def load_data_offenseval(samples_path, labels_path):
    df = pd.read_csv(os.path.join(samples_path), delimiter='\t', encoding='utf-8')
    labs = pd.read_csv(os.path.join(labels_path), delimiter='\t', encoding='utf-8')
    df = df.drop(df.columns[2:5], axis=1)
    labs = labs.drop(labs.columns[0], axis=1)
    labs = labs.replace({'NOTABU': 'non-toxic', 'EXP': 'toxic', 'IMP': 'toxic'})
    df = pd.concat([df, labs], axis=1)
    df = df.drop(df.columns[0], axis=1)
    return df

def load_data_perturbed(path):
    df = pd.read_csv(os.path.join(path), delimiter='\t', encoding='utf-8')
    df = df.drop(df.columns[0], axis=1)
    return df

c:\Users\eviev\Documents\Python\Schoolwork2\schoolwork2\lib\site-packages\transformers\pipelines\text_classification.py:105: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


#### Model Predictions + Shap & LIME implementations

##### Shap Implementation

In [227]:
'''
Run shap and save top 5 critical words for toxic prediction.
Inputs:
- data;     dataset to evaluate
- pipeline; model pipeline
- savename; file name used for saving results (without extension)
'''

def run_shap(data, pipeline, savename='data_shap'):
    # Explainer
    shap_explainer = shap.Explainer(pipeline, algorithm='partition')

    # Gather top 5 critical words (max score; positive = more toxic)
    shap_expl = shap_explainer(data[data.columns[0]].values.tolist())

    with open(f'{savename}_shapobjects.pkl', 'wb') as f:
        pickle.dump(shap_expl, f)

    def save_tokenscores(shapresults):
        data_shap = []
        for sample_expl in shapresults:
            scores = sample_expl.values[:,1]  # toxicity contributing scores
            tokens = sample_expl.data       # feature tokens
            data_shap.append(list(zip(tokens, scores)))
        with open(f'{savename}.json', 'w') as f:
            json.dump(data_shap, f)

    save_tokenscores(shap_expl)

##### LIME Implementation + model predictions

In [228]:
'''
Run model predictions and LIME and save top 5 critical words for toxic prediction
Inputs:
- data;         dataset to evaluate
- model;        model used for predictions
- tokenizer;    tokenizer used by the model
- savename;     file name used for saving results (without extension)
'''

def run_lime(data, model, tokenizer, savenameLIME, savenameMODEL):
    # Use the label names from your config.json
    class_names = ['non-toxic', 'toxic']

    # Explainer
    lime_explainer = LimeTextExplainer(class_names=class_names)

    # model predictor
    def model_predict(texts):
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}  # Move inputs to GPU
        model.to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        return probs.detach().cpu().numpy()


    data_lime = []
    data_model = []
    iterations = data.shape[0]
    for index, row in data.iterrows():
        (sample, label) = row.to_numpy()
        
        # Analyze top 5 critical words (max score; positive = more toxic)
        lime_expl = lime_explainer.explain_instance(sample, model_predict, num_features=10, num_samples=100)
        data_lime.append(sorted(lime_expl.as_list(label=1), key=lambda tup: tup[1], reverse=True)[:5])
        
        (nontoxic, toxic) = lime_expl.predict_proba
        if toxic > nontoxic:
            data_model.append('toxic')
        else:
            data_model.append('non-toxic')
        
        if index % 10 == 0: print(f'\r[{((index/iterations)*100):>6.2f}%]', end='')

    with open(f'{savenameLIME}.json', 'w') as f:
        json.dump(data_lime, f)
    
    with open(f'{savenameMODEL}.json', 'w') as f:
        json.dump(data_model, f)

#### Perturbed Dataset Generation

##### LIME perturbations (sub_c)
Applies one character perturbation to a random character in each top 5 critical word for toxic prediction. Perturbations are only applied to by the model toxic predicted samples.

In [229]:
''' 
sub_c word perturbation function.
Applies a random perturbation to a random char_map available
character in the input word. The amount of sub_c word perturbations
can be increased by increasing the amount parameter.
char_map was made by intuition based on internet culture knowledge
and taking into consideration easily accessible characters by keyboard keys.
'''

def perturb_word_lime(word, amount):
    char_map = {
        'a': ['4', '@'],
        'b': ['8', '6'],
        'c': ['<', '(', '{', '['],
        #'d': [],
        'e': ['3'],
        #'f': [],
        'g': ['9', '6'],
        #'h': [],
        'i': ['l', '1', '!'],
        'j': ['i'],
        #'k': [],
        'l': ['I', '1', '!'],
        #'m': [],
        #'n': [],
        'o': ['0', '()', '{}'],
        #'p': [],
        'q': ['9'],
        #'r': [],
        's': ['5', '$'],
        't': ['7', '+'],
        'u': ['v'],
        'v': ['u'],
        'w': ['vv'],
        'x': ['ks'],
        'y': ['i'],
        'z': ['s', '2']
    }
    l = len(word)
    if l < 2:
        return word # don't perturb non-words
    perturbed_chars = []
    for n in range(amount):
        idx = random.randrange(0, l-1)
        count = 0
        while idx in perturbed_chars or word[idx] not in char_map:
            count += 1
            idx += 1
            if idx >= l:
                idx -= l
            if count >= l:
                return word
        word = word[:idx] + random.choice(char_map[word[idx]]) + word[idx+1:]
        if l != len(word):
            perturbed_chars = [x+1 if x>=idx else x for x in perturbed_chars]
            perturbed_chars.extend([idx, idx+1])
            l = len(word)
        else: perturbed_chars.append(idx)
    return word

In [230]:
'''
Create new dataset where for each sample of the input dataset, 
the LIME evaluated top 5 critical words are perturbed 
with one sub_c word perturbation each.
This function loads two files:
- the LIME data  (toxic prediction top 5 critical words per sample)
- the model data (model predicted label per sample)

Inputs:
- data;     dataset to evaluate
- savename; file name used for saving dataset (without extension)
'''

def generate_dataset_lime(data, savename='dataset_limed'):
    pert_data = data.copy(deep=True)

    data_lime = read_lime('data_lime')
    data_model = read_model('data_model')

    for idx, words in enumerate(data_lime):
        if data_model[idx] == 'toxic': # if the prediction is toxic, then perturb data sample to fool model
            text = pert_data[pert_data.columns[0]][idx]
            for wordscore in words:
                word = wordscore[0]
                pert_word = perturb_word_lime(word, 1) # perturb 1 character if possible
                word_occurences = re.finditer(rf'\b{word}\b', text)
                for x in reversed(list(word_occurences)):
                    text = text[:x.start()] + pert_word + text[x.end():]
            pert_data.loc[idx, pert_data.columns[0]] = text

    display(data[:6])
    display(pert_data[:6])

    pert_data.to_csv(f'{savename}.tsv', sep="\t") 

##### Shap Perturbations

In [ ]:
''' 
sub_w word perturbation function.
Changes SHAP top 5 toxic critical words to similar words.
'''

class perturb_word_shap():
    def __init__(self):
        # model glove uncomment till self.aug = naw.SynonymAug, then comment self.aug = naw.SynonymAug line
        # from nlpaug.util.file.download import DownloadUtil
        # DownloadUtil.download_glove(model_name='glove.6B', dest_dir='.') # Download GloVe model
        # self.aug = naw.WordEmbsAug(
        #     model_type='glove', 
        #     model_path=os.path.join(os.getcwd(), 'glove_model', 'glove.6B.300d.txt'),
        #     action="substitute",
        #     top_k=5)
        self.aug = naw.SynonymAug(aug_src='wordnet',) # using synonyms
    def perturb(self, word):
        word = self.aug.augment(word)
        return word

In [232]:
'''
Generate perturbed dataset SHAP.
This function loads two files:
- the SHAP data  (toxic prediction top 5 critical words per sample)
- the model data (model predicted label per sample)

Inputs:
- data;     dataset to evaluate
- savename; file name used for saving dataset (without extension)

! This model outputs tokens that it couldn't perturb either due to:
1. the token being a part of a word.
2. the token being a symbol.
3. the token being an emoji.
'''

def generate_dataset_shap(data, savename='dataset_shapped'):
    pert_data = data.copy(deep=True)

    data_shap = read_shap('data_shap')
    data_model = read_model('data_model')

    print('shap data length: ', len(data_shap))
    print('model data length: ', len(data_model))

    # initialize shap perturb model
    shaperturber = perturb_word_shap()

    print("Some tokens don't have perturbations or are a part of a word. These will have their information printed. They are not perturbed.")
    for idx, words in enumerate(data_shap):
        if data_model[idx] == 'toxic': # if the prediction is toxic, then perturb data sample to fool model
            text = pert_data[pert_data.columns[0]][idx]
            perturbed_text = text
            word_list = [word for (word, score) in words]
            indexed_wordscores = [(i, word, score) for i, (word, score) in enumerate(words)]
            top5 = sorted(indexed_wordscores, key=lambda item: item[1], reverse=True)[:5]
            top5 = sorted(top5, key=lambda item: item[0], reverse=True)
            for (i, word, _) in top5:                                     # for every word in top 5 critical words according to shap:
                word = word.strip()                                         # make sure there are no leading or trailing spaces
                if word in ['USER', 'URL']: break
                pert_word = shaperturber.perturb(word)                      # perturb 1 word if possible (otherwise return same word)
                if pert_word == []: pert_word = word
                else: pert_word = pert_word[0]
                occurence = word_list[:i].count(word)                     # find the occurence of this word in the text (is it the first, or second, or .. in text)
                word_occurences = list(re.finditer(rf'\b{word}\b', text))   # find all occurences of word (exact word search)
                if word_occurences:                                         # needed because re.finditer(emoji) returns empty list
                    try:
                        x = word_occurences[occurence]                          # get the exact word in text (position, span, etc)
                        perturbed_text = perturbed_text[:x.start()] + pert_word + perturbed_text[x.end():]    # replace with perturbation
                    except:
                        print("\n\n-----------------------------------")   
                        print(text)
                        print(word_list)                                     
                        print(word)
                        print(word_occurences)
                        print(occurence)
            pert_data.loc[idx, pert_data.columns[0]] = perturbed_text     # save perturbed sample

    display(data[:6])
    display(pert_data[:6])

    pert_data.to_csv(f'{savename}.tsv', sep="\t") 

#### Result Gathering and Analysis

In [233]:
'''
Get the base statistics TP/FP/FN/TN and precision/recall/F1-score using ground truth, model predictions, and the positive class
Calculate Precision / Recall / F1-scores using statistics table
'''

def get_model_performance(gtruth, pred, pclass):
    if len(pred) != len(gtruth): 
        print("ERROR: inputs do not match in size!")
        return

    TP=0; FP=0; FN=0; TN=0
    for idx, lab_pred in enumerate(pred):
        if lab_pred == pclass:
            if gtruth[idx] == pclass: TP+=1
            else: FP+=1
        else:
            if gtruth[idx] == pclass: FN+=1
            else: TN+=1
    base_statistics = np.array([[TP, FP], [FN, TN]])
    precision       = base_statistics[0, 0] / base_statistics.sum(axis=1)[0]
    recall          = base_statistics[0, 0] / base_statistics.sum(axis=0)[0]
    F1              = base_statistics[0, 0] / (base_statistics[0, 0] + .5*(base_statistics[0, 1] + base_statistics[1, 0]))
    return (base_statistics, precision, recall, F1)

# print the result in a nice to read format
def print_analysis_result(perf_data, pclass):
    print("--------------------------------------------------")
    print(f"Performance Data for Positive Class: {pclass}.")
    print("\nAnalysis Matrix:\n", perf_data[0])
    print("\nPrecision:\n", perf_data[1])
    print("\nRecall:\n", perf_data[2])
    print("\nF1-score:\n", perf_data[3])
    print("--------------------------------------------------\n\n")



In [234]:
'''
Generate analysis output.
Count TP/FP/FN/TN & calculate precision/recall/F1-score/macro-F1

Inputs:
- data;     dataset to evaluate
- predicted_labels; predicted labels (same order as gtruth)
- class_names; the class names in the predictions (order matters)
- savename; file name used for saving dataset (without extension)
'''

def run_analysis(data, predicted_labels, class_names, savename):
    # buffer to save cell output
    buf = io.StringIO()

    gtruth = data[data.columns[1]].to_numpy() # ground truth labels

    performance_nontoxic = get_model_performance(gtruth=gtruth, pred=predicted_labels, pclass=class_names[0])
    performance_toxic    = get_model_performance(gtruth=gtruth, pred=predicted_labels, pclass=class_names[1])

    with redirect_stdout(buf):
        print(f"{savename}")
        print_analysis_result(performance_nontoxic, 'non-toxic')
        print_analysis_result(performance_toxic, 'toxic')
        print("macro F1-score:\n", (performance_nontoxic[3]+performance_toxic[3])/2)

    # Get the captured output as a string
    with open(f'{savename}.txt', 'w') as f:
        f.write(buf.getvalue())


#### Running the Anaysis
Here all the previously defined functions are used to run the analysis.  
  
We've tried reducing the time it takes as much as possible, for us but also for others running this code. It will still take a while.  
For LIME it takes about 15 minutes with cuda enabled on an NVIDIA RTX3090 GPU.  
For SHAP it takes several hours on the same setup (24-28 hours).  
Both processes can be sped up by splitting the data into multiple parts.

In [239]:
'''
params: model_path, offenseval_path, abuseval_path, samples_path, labels_path
params: model, tokenizer, pipeline
pert_data = load_data_perturbed('dataset_shapped_synonym.tsv')
'''
class_names = ['non-toxic', 'toxic'] # class names (double check if order is correct)

# FIRST LOAD DATA:
data = load_data_offenseval(samples_path, labels_path)

# display head & tail
display(data.head())
display(data.tail())


# RUN LIME ANALYSIS:

# lime & default dataset
#run_lime(data=data, model=model, tokenizer=tokenizer, savenameLIME='data_lime', savenameMODEL='data_model')
generate_dataset_lime(data=data, savename='dataset_limed') 
run_analysis(data=data, predicted_labels=read_model('data_model'), class_names=class_names, savename='analysis_defaults')
#run_lime(data=load_data_perturbed('dataset_limed.tsv'), model=model, tokenizer=tokenizer, savenameLIME='data_lime', savenameMODEL='data_model_perturbed_lime')
run_analysis(data=data, predicted_labels=read_predictions_perturbed('lime'), class_names=class_names, savename='analysis_lime_perturbed')

# shap
run_shap(data=data, pipeline=pipeline, savename='data_shap')
generate_dataset_shap(data=data, savename='dataset_shapped.json')
run_lime(data=load_data_perturbed('dataset_shapped.tsv'), model=model, tokenizer=tokenizer, savenameLIME='data_lime', savenameMODEL='data_model_perturbed_shap')
run_analysis(data=data, predicted_labels=read_predictions_perturbed('shap'), class_names=class_names, savename='analysis_shap_perturbed')

,tweet,abuse
0,@USER She should ask a few native Americans wh...,non-toxic
1,@USER @USER Go home you’re drunk!!! @USER #MAG...,non-toxic
2,Amazon is investigating Chinese employees who ...,non-toxic
3,"@USER Someone should'veTaken"" this piece of sh...",non-toxic
4,@USER @USER Obama wanted liberals &amp; illega...,non-toxic


,tweet,abuse
13235,@USER Sometimes I get strong vibes from people...,toxic
13236,Benidorm ✅ Creamfields ✅ Maga ✅ Not too sh...,non-toxic
13237,@USER And why report this garbage. We don't g...,toxic
13238,@USER Pussy,non-toxic
13239,#Spanishrevenge vs. #justice #HumanRights and ...,non-toxic


,tweet,abuse
0,@USER She should ask a few native Americans wh...,non-toxic
1,@USER @USER Go home you’re drunk!!! @USER #MAG...,non-toxic
2,Amazon is investigating Chinese employees who ...,non-toxic
3,"@USER Someone should'veTaken"" this piece of sh...",non-toxic
4,@USER @USER Obama wanted liberals &amp; illega...,non-toxic
5,@USER Liberals are all Kookoo !!!,toxic


,tweet,abuse
0,@USER She should ask a few native Americans wh...,non-toxic
1,@USER @USER Go home you’re drunk!!! @USER #MAG...,non-toxic
2,Amazon is investigating Chinese employees who ...,non-toxic
3,"@USER Someone sh()uld'veTaken"" this plece of 5...",non-toxic
4,@USER @USER Obama wanted liberals &amp; illega...,non-toxic
5,@USER Liber4ls ar3 4ll Ko0koo !!!,toxic


c:\Users\eviev\Documents\Python\Schoolwork2\schoolwork2\lib\site-packages\transformers\pipelines\base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(








































KeyboardInterrupt: 